# 第21章　眼科②：OCTと層構造ガイド型AI

**『医療診断支援AI開発　第2巻　応用編 ― 本格実装から臨床導入まで（第2巻　応用編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-vol2

## 21.5　マルチチャネル入力の実装

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

class LayerGuidedClassifier(nn.Module):
    def __init__(self, num_classes, in_channels=16):
        super().__init__()
        # EfficientNet-B3を出発点にする
        base = models.efficientnet_b3(weights="IMAGENET1K_V1")
        # 最初の畳み込みを16チャネル入力に置き換える。
        # ここでConvを新たに初期化すると、ImageNet事前学習の重みを捨ててしまう。
        # RGB3chぶんは元の重みをそのまま引き継ぎ、増やした13chはその平均で初期化する。
        old_conv = base.features[0][0]
        new_conv = nn.Conv2d(
            in_channels, old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=False,
        )
        with torch.no_grad():
            w = old_conv.weight                       # (out, 3, k, k)
            new_conv.weight[:, :3] = w                # RGBぶんは事前学習の重みを継承
            new_conv.weight[:, 3:] = w.mean(dim=1, keepdim=True)   # 残りは平均で初期化
        base.features[0][0] = new_conv
        self.backbone = base.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Linear(1536, num_classes)

    def forward(self, x):   # x: (B, 16, H, W)
        x = self.backbone(x)
        x = self.pool(x).flatten(1)
        return self.head(x)

# 入力の組み立て：OCT画像(3ch) + 層構造マップ(13ch) = 16ch
def build_input(oct_rgb, layer_maps):
    return torch.cat([oct_rgb, layer_maps], dim=1)  # チャネル方向に結合

## 層厚マップを描く：ETDRSグリッドと9セクター

In [ ]:
import numpy as np
def thickness_map(ilm_z, rpe_z, z_res_um):
    # z_res_um は軸方向のピクセル間隔[µm/画素]。装置の光学的な分解能ではない
    return (rpe_z - ilm_z) * z_res_um            # (H, W) の網膜厚マップ[µm]

def etdrs_sectors(tmap, fovea_xy, mm_per_px_xy, x_positive_is, y_positive_is="inferior"):
    """mm_per_px_xy: (x方向, y方向) の1画素あたりmm。黄斑キューブはAスキャン方向と
    Bスキャン方向でピクセル間隔が大きく違う（例：6×6mmを512×128で撮る）ため、
    等方と仮定すると外輪の上下セクターが空になる。
    x_positive_is: 画像のx正方向が "temporal"(耳側) か "nasal"(鼻側) か。
    右眼(OD)と左眼(OS)で反転するので、自分のデータで必ず確認して渡すこと。
    y_positive_is: 画像のy正方向（配列の下向き）が "inferior"(下方) か "superior"(上方) か。
    書き出しの規約で上下が反転していることがあるので、これも確かめて渡す。
    グリッドの一部が撮像範囲の外にかかるセクターは、値を None にする（はみ出した部分を除いた平均は返さない）。
    この例では欠損の補間や部分平均は行わない方針とする。
    返り値は (セクターごとの平均厚の辞書, 計測できなかったセクターの理由の辞書)。一部でも厚みが欠損したセクターは値を None にし、理由を添える。"""
    if x_positive_is not in ("temporal", "nasal"):
        raise ValueError("x_positive_is は 'temporal' か 'nasal'")
    if y_positive_is not in ("inferior", "superior"):
        raise ValueError("y_positive_is は 'inferior' か 'superior'")
    H, W = tmap.shape
    mm_x, mm_y = mm_per_px_xy
    if not all(np.isfinite(v) and v > 0 for v in (mm_x, mm_y)):
        raise ValueError("ピクセル間隔は正の有限値にする")
    if not (0 <= fovea_xy[0] < W and 0 <= fovea_xy[1] < H):
        raise ValueError("中心窩の座標が画像の外")
    # 6×6mmのキューブでは、中心窩が数画素ずれるだけで直径6mmのグリッドが撮像範囲からはみ出す。
    # 画像全体をエラーで止めずに、はみ出したセクターだけを計測不能にするため、座標を画像の外まで広げて数える。
    px, py = int(np.ceil(3.0 / mm_x)) + 1, int(np.ceil(3.0 / mm_y)) + 1
    yy, xx = np.mgrid[-py:H + py, -px:W + px]
    in_fov = (xx >= 0) & (xx < W) & (yy >= 0) & (yy < H)
    tmap = np.pad(np.asarray(tmap, dtype=float), ((py, py), (px, px)), constant_values=np.nan)
    dx = (xx - fovea_xy[0]) * mm_x                # 先に物理mmへ直してから
    dy = (yy - fovea_xy[1]) * mm_y                # 半径と角度を計算する
    r = np.hypot(dx, dy)
    ang = np.arctan2(dy, dx)
    rings = {"central": r <= 0.5, "inner": (r > 0.5) & (r <= 1.5),
             "outer": (r > 1.5) & (r <= 3.0)}
    # 耳側(T)/鼻側(N)は左右眼で反転する。x正方向がどちらを向くかは、
    # 左右眼と機器の書き出し規約で決まるため、呼び出し側が明示的に渡す。
    horiz_pos = "T" if x_positive_is == "temporal" else "N"   # x正方向のセクター
    horiz_neg = "N" if x_positive_is == "temporal" else "T"   # x負方向のセクター
    vert_pos = "I" if y_positive_is == "inferior" else "S"    # y正方向（配列の下向き）のセクター
    vert_neg = "S" if y_positive_is == "inferior" else "I"    # y負方向のセクター
    # 境界を半開区間にそろえ、各画素を必ず一つの象限だけへ割り当てる。
    quad = {
        vert_neg: (ang >= -3*np.pi/4) & (ang < -np.pi/4),
        vert_pos: (ang >= np.pi/4) & (ang < 3*np.pi/4),
        horiz_pos: (ang >= -np.pi/4) & (ang < np.pi/4),
        horiz_neg: (ang >= 3*np.pi/4) | (ang < -3*np.pi/4),
    }
    valid = np.isfinite(tmap)
    out, why = {}, {}
    def put(name, mask):
        if not mask.any():
            out[name] = None
            why[name] = "セクター内の画素がない（ピクセル間隔を確認）"
        elif not in_fov[mask].all():
            out[name] = None
            why[name] = "セクターの一部が撮像範囲の外にある"
        elif not valid[mask].all():
            out[name] = None
            why[name] = "セクター内に厚みの欠損がある"
        else:
            out[name] = float(tmap[mask].mean())
    put("central", rings["central"])
    for r_name in ("inner", "outer"):
        for q_name, q in quad.items():
            put(f"{r_name}_{q_name}", rings[r_name] & q)
    return out, why                              # 9セクターの平均厚と、計測できなかった理由